"""
This code is provided as supplemental material to the publication "Machine learning for small data sets: an exemplary study on the classification of highly complex surface micromorphologies" 
by M. Henkel, M. Sprenger, and O. Lieleg submitted to Materials Today Advances on October 17th, 2025.

"""

In [ ]:
import sklearn as skl
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.model_selection import train_test_split
from NETCORE import *
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
def mixed_classifier(make_prediction = True, Class = 'Class_2'): 
    #make_prediction: set 'True' to classify unknown dataset
    #Class = 'Class_1'(damage order sensitive) 'Class_2' (damage order insensitive)
    
    # Import data
    pooled_labelled, label = importdata(prediction_import =False,  Class=Class, delimiter = ';', decimal = ',')
    
    # Feature elimination
    features = feature_elimination(pooled_labelled, t_corr = 0.8)         
    print('\nFeature elimination using NETCORE algorithm by Rickert et al. 2022')
        

    # Create dataset of features: X & Y
    X = pooled_labelled[features]
    Y = label
    
    # Scaling of dataset (X)
    X_scaled, scalers = scaling(X)
    
    clf1 = skl.ensemble.RandomForestClassifier(n_estimators=300, min_impurity_decrease=0)

    classifier = ['RandForest', clf1]
    
    # Fit RF classifier
    global TestTrain_results
    TestTrain_results, clf_fitted = train_and_evaluate_random_forest(X_scaled, Y, classifier, features) 
    classifier[1] = clf_fitted
    
    # Predict independent dataset
    if make_prediction:
        
        global prediction_results
        prediction_results = prediction_function(classifier, features, scalers, Class)
        

In [ ]:
def importdata(prediction_import,  Class,  delimiter , decimal):
    print('\nImporting data')

    # Specify file path
    filenames = ['Data/ABR_surface_param.csv', 'Data/ADH_surface_param.csv','Data/ERO_surface_param.csv','Data/NOD_surface_param.csv','Data/ABR_ADH_surface_param.csv','Data/ABR_ERO_surface_param.csv', 'Data/ADH_ABR_surface_param.csv', 'Data/ADH_ERO_surface_param.csv', 'Data/ERO_ABR_surface_param.csv', 'Data/ERO_ADH_surface_param.csv']  
    filenames_predict =  ['Holdout_data/ABR_surface_param.csv', 'Holdout_data/ADH_surface_param.csv','Holdout_data/ERO_surface_param.csv','Holdout_data/NOD_surface_param.csv', 'Holdout_data/ABR_ADH_surface_param.csv','Holdout_data/ABR_ERO_surface_param.csv','Holdout_data/ERO_ADH_surface_param.csv']
   
    if prediction_import:
        filenames = filenames_predict

    # Save the data into a list and convert it into pandas DataFrame
    collect_df = []
    for name in filenames:
        print ('pooling class: %s comprising %i samples' %(name, pd.read_csv(name, delimiter = delimiter, decimal = decimal, engine ='python').shape[0]))
        collect_df.append(pd.read_csv(name, delimiter = delimiter, decimal = decimal, engine ='python'))
          
    global pooled_dataset
    pooled_dataset = pd.concat (collect_df, ignore_index = True)
    pooled_dataset = pooled_dataset.dropna()
    
    # Saving the labels according to Class:
    if Class == 'Class_2':
        label = pooled_dataset['Class_2']
    elif Class == 'Class_1':
        label = pooled_dataset['Class_1']
    else:
        label = None
        
        
    # Dropping label information and uncessesary columns
    columns_to_drop = ['File name','Area size','Class_2', 'Class_1']
    for column in columns_to_drop:
        if column in pooled_dataset.columns:
            pooled_dataset = pooled_dataset.drop(column, axis=1)
    
    return pooled_dataset, label

In [ ]:
def feature_elimination(pooled_labelled, t_corr, export_to_CSV = False):  
    print('\nPerfoming feature elimination using NETCORE algorithm')
    
    # 0 ≤ t_corr ≤ 1: defines the 'level' of correlation that is allowed between the features. High values: less information loss but risk of redundancies, small values: less redundancies but risk of information loss.
    # Run NETCORE algorithm
    reduced_vector = run_NETCORE(pooled_labelled, t_corr)
    
    # Export the reduced data as CSV:
    if export_to_CSV:
        pooled_labelled[reduced_vector].to_csv('NETCORE data set.csv',index=False )

    return reduced_vector

In [ ]:
def scaling (X, scaling = True):
    print('\nPerfoming scaling of dataset')
    
    if scaling is False:
            print('Scaling is deactivated')
            X = X.select_dtypes(include=[np.number]).to_numpy()
            X_scaled = X
            scalers = []

            return X_scaled, scalers

    else:
            # Create scalers
            QuantileTrans = QuantileTransformer(output_distribution = 'uniform', random_state=42)
            MinMax = MinMaxScaler()
            scalers =  [['QuantileTransformer', QuantileTrans], ['MinMax', MinMax]]

            # Scaling 
            X_scaled = X
            for scaler in scalers:
                print('Preprocessing: Applying %s' % (scaler[0]))
                X_scaled = scaler[1].fit_transform(X_scaled)
                 
            return X_scaled, scalers

In [ ]:
def train_and_evaluate_random_forest(X_scaled, Y, classifier, features, test_size = 0.3):
    print('\nPerfoming model training and evaluation with Test/Train Split')

    # Unpacking of classifier 
    clf_name = classifier[0]
    clf = classifier[1]
    
    # Split data into training and test sets
    X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=test_size, random_state=1) 
        
    # Train the model
    model = clf.fit(X_train, Y_train)
    
    # Make predictions on the test set
    Y_pred = clf.predict(X_test)
        
    # Calculate feature importance
    feature_importance = clf.feature_importances_
    feature_importance_map = list(zip(features, feature_importance))
    
    # Generate classification report
    report = metrics.classification_report(Y_test, Y_pred)
    
    # Print results
    print('\nFeature importances for Random Forest classifier:\n', feature_importance_map)
    print('\nReport for Random Forest classifier:\n', report)
    
    # Create DataFrame with results
    results = pd.DataFrame()
    pd_Y_test = pd.Series(Y_test).reset_index(drop=True)
    pd_Y_pred = pd.Series(Y_pred).reset_index(drop=True)
    results['Y_real'] = pd_Y_test
    results['Y_pred'] = pd_Y_pred
    
    # Confusion matrix
    # Combine training and predicted labels, then find unique values
    all_labels = np.unique(np.concatenate((Y_test, Y_pred)))
    conf_matrix = metrics.confusion_matrix(Y_test,Y_pred, labels = all_labels)
    disp = metrics.ConfusionMatrixDisplay(confusion_matrix = conf_matrix, display_labels = all_labels)
    disp.plot()
    plt.savefig('Confusion Matrix_Test-Train.eps', 
                format='eps', 
                dpi=300, 
                bbox_inches='tight',
                pad_inches=0.1)
    plt.show()
    
    return results, clf


In [ ]:
def prediction_function(classifier, features, scalers, Class):
    print('\nPrediction of user input using %s classifier:' %classifier[0])
    
    # Reading in the data
    print('\nImporting data')
    pooled_labelled_predict, label_predict = importdata(prediction_import=True, Class=Class, delimiter = ';', decimal = ',')
    
    # Define X
    X_predict = pooled_labelled_predict[features]
    
    # Define Y (if available in the data, otherwise set to None)
    Y = label_predict
    
    # Scaling of X_predict
    X_predict_scaled = X_predict
    for scaler in scalers:
        X_predict_scaled = scaler[1].transform(X_predict_scaled)
        
    # Make prediction
    clf = classifier[1]
    predictions = clf.predict(X_predict_scaled)
    print('\nMaking predictions')

    # Analyze prediction decisions
    report = metrics.classification_report(Y, predictions) if Y is not None else None
    print('Classification Report for Prediction:\n', report)
    
    # Create a result pd.DataFrame comprising the individual predictions
    results = pd.DataFrame()
    results['Predicted_Label'] = predictions
    if Y is not None:
        results['Real_Label'] = Y
        
    return results
        

In [ ]:
mixed_classifier()